# 03 — Fmr1 | Within-genotype heatmaps

**Prepared, not yet computed.** This notebook will only load per-mouse state metrics produced by notebook 02 and generate 6×6 paired-state comparisons; minimum five independent mice per test.


In [1]:
# Runtime setup — run at the start of a fresh Colab session
%pip -q install one-api ibllib iblatlas pandas numpy scipy matplotlib statsmodels pyarrow
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from one.api import ONE
import pandas as pd
import numpy as np
ROOT = Path('/content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned')
assert ROOT.exists(), f'Cannot find project root: {ROOT}'
one = ONE(base_url='https://openalyx.internationalbrainlab.org', username='intbrainlab', password='international', silent=True)
print('Project root:', ROOT)

GENOTYPE = 'Fmr1'
METRICS_FILE = ROOT / '04_results' / GENOTYPE / 'electrophysiology_v1' / f'Fmr1_mouse_region_phase_metrics.csv'
print('Analysis file:',METRICS_FILE)
if METRICS_FILE.exists():
    frame = pd.read_csv(METRICS_FILE)
    print('Mice:',frame.subject.nunique(), 'Rows:',len(frame))
else:
    print('Not yet computed. First run the genotype-specific 02_process_and_cache notebook.')


Mounted at /content/drive
Project root: /content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned
Analysis file: /content/drive/MyDrive/Reanalysis/Our_Analysis/Behavior_aligned/04_results/Fmr1/electrophysiology_v1/Fmr1_mouse_region_phase_metrics.csv
Mice: 15 Rows: 864


In [2]:

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
from IPython.display import display

ROOT = Path(
    "/content/drive/MyDrive/Reanalysis/"
    "Our_Analysis/Behavior_aligned"
)

DATA = ROOT / "04_results/Fmr1/electrophysiology_v1"
df = pd.read_csv(DATA / "Fmr1_mouse_region_phase_metrics.csv")

REGIONS = ["DG", "CA1", "CA3", "SUB", "VISp",
           "MOs", "CP", "MRN", "APN"]

STATES = ["Q", "S", "M", "F_reward", "F_error", "I"]

omnibus = []
posthoc = []

for region in REGIONS:
    sub = df[df["region"] == region]

    wide = sub.pivot_table(
        index="subject",
        columns="state",
        values="fr_hz",
        aggfunc="mean"
    ).reindex(columns=STATES).dropna()

    n = len(wide)
    if n < 5:
        continue

    values = wide.to_numpy()

    if np.all(values == values[:, :1]):
        chi2, p = 0.0, 1.0
    else:
        chi2, p = friedmanchisquare(
            *(wide[s].values for s in STATES)
        )

    omnibus.append({
        "region": region, "n_mice": n,
        "friedman_chi2": chi2,
        "kendall_W": chi2 / (n * (len(STATES) - 1)),
        "p_raw": p
    })

    pairs = []
    for a, b in combinations(STATES, 2):
        d = wide[b] - wide[a]
        if np.all(d.to_numpy() == 0):
            p_pair = 1.0
        else:
            p_pair = wilcoxon(
                wide[a], wide[b],
                alternative="two-sided"
            ).pvalue

        pairs.append({
            "region": region,
            "comparison": f"{a} vs {b}",
            "difference_hz": d.mean(),
            "p_raw": p_pair
        })

    corrected = multipletests(
        [x["p_raw"] for x in pairs],
        method="holm"
    )[1]

    for record, adjusted in zip(pairs, corrected):
        record["p_holm"] = adjusted
        posthoc.append(record)

omnibus = pd.DataFrame(omnibus)
posthoc = pd.DataFrame(posthoc)

omnibus["p_fdr"] = multipletests(
    omnibus["p_raw"], method="fdr_bh"
)[1]

print("FRIEDMAN — ALL REGIONS")
display(omnibus.sort_values("p_fdr").round(5))

print("SIGNIFICANT WILCOXON COMPARISONS (HOLM)")
significant = posthoc[posthoc["p_holm"] < 0.05]
display(significant.sort_values("p_holm").round(5))

omnibus.to_csv(DATA / "Fmr1_FR_Friedman.csv", index=False)
posthoc.to_csv(DATA / "Fmr1_FR_Wilcoxon_posthoc.csv", index=False)


FRIEDMAN — ALL REGIONS


,region,n_mice,friedman_chi2,kendall_W,p_raw,p_fdr
1,CA1,13,42.89011,0.65985,0.00000,0.00000
5,MOs,12,42.14286,0.70238,0.00000,0.00000
4,VISp,13,35.28571,0.54286,0.00000,0.00000
7,MRN,12,30.14286,0.50238,0.00001,0.00003
6,CP,10,24.62857,0.49257,0.00016,0.00030
0,DG,15,23.83810,0.31784,0.00023,0.00035
3,SUB,11,21.64935,0.39362,0.00061,0.00078
8,APN,12,19.95238,0.33254,0.00128,0.00144
2,CA3,10,5.42857,0.10857,0.36584,0.36584


SIGNIFICANT WILCOXON COMPARISONS (HOLM)


,region,comparison,difference_hz,p_raw,p_holm
10,DG,M vs F_error,-0.88339,0.00006,0.00092
15,CA1,Q vs S,0.90059,0.00024,0.00366
16,CA1,Q vs M,2.60276,0.00024,0.00366
19,CA1,Q vs I,0.68258,0.00024,0.00366
20,CA1,S vs M,1.70217,0.00024,0.00366
26,CA1,M vs I,-1.92019,0.00049,0.00537
18,CA1,Q vs F_error,1.84472,0.00049,0.00537
62,VISp,Q vs F_reward,2.18698,0.00049,0.00732
107,MRN,Q vs F_reward,6.26700,0.00049,0.00732
76,MOs,Q vs M,1.96332,0.00049,0.00732
